# Notebook 1: Business Understanding & Data Profiling

## What is Customer Churn?

**Customer churn** (also known as customer attrition) occurs when a customer stops doing business with a company. In the telecom industry, this means a subscriber cancels their service or switches to a competitor.

## Why Does Churn Matter?

Churn is one of the most expensive problems in subscription-based businesses:

- **Acquiring a new customer costs 5–25x more** than retaining an existing one (Harvard Business Review)
- A **5% increase in retention** can boost profits by **25–95%**
- In the US telecom industry, average annual churn rates range from **15% to 25%**
- Each churned customer represents **lost lifetime value (LTV)** — often hundreds or thousands of dollars

## What This Project Does

This project builds a **machine learning pipeline** to:

1. **Identify** customers who are at high risk of churning
2. **Understand** which factors drive churn (contract type, tenure, charges, etc.)
3. **Generate** personalized retention recommendations for at-risk customers
4. **Quantify** the business value of proactive intervention

### Dataset
We use the **IBM Telco Customer Churn** dataset, which contains information about 7,043 customers including their demographics, account details, services subscribed, and whether they churned.

### Notebooks in This Series
| Notebook | Topic |
|----------|-------|
| 01 | Business Understanding & Data Profiling |
| 02 | Exploratory Data Analysis (EDA) |
| 03 | Feature Engineering |
| 04 | Model Training & Comparison |
| 05 | Model Explainability & SHAP Analysis |

## Setup

We add the `src/` directory to the Python path so we can import our custom modules for loading configuration and raw data.

In [ ]:
import pandas as pd
import numpy as np
import yaml
import sys
import os

# Add the src directory to the Python path
sys.path.insert(0, os.path.abspath('../src'))

from data_preprocessing import load_raw_data

def load_config(path='../config.yaml'):
    """Load project configuration from YAML file."""
    with open(path, 'r') as f:
        return yaml.safe_load(f)

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Load raw data
RAW_DATA_PATH = '../data/raw/telco_churn.csv'

df = load_raw_data(RAW_DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Number of rows (customers): {df.shape[0]:,}')
print(f'Number of columns (features): {df.shape[1]}')
print(f'\nColumn names:')
print(df.columns.tolist())

## Dataset Overview

The Telco Customer Churn dataset has **21 columns** covering four categories:

- **Customer Demographics**: `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- **Account Info**: `customerID`, `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`
- **Services**: `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- **Target**: `Churn` (Yes / No)

In [ ]:
# General info about the dataset
print('=== DataFrame Info ===')
df.info()

print('\n=== First 5 Rows ===')
display(df.head(5))

print('\n=== Descriptive Statistics ===')
display(df.describe(include='all'))

## Missing Values & Data Quality

Before building models, we need to understand the quality of our data. Common issues include:

- **Missing values**: NaN entries that need to be imputed or removed
- **Duplicate rows**: Repeated records that could bias the model
- **Incorrect data types**: For example, `TotalCharges` is stored as a string in the raw data because some entries are blank spaces rather than proper NaN values

We will handle these issues in the preprocessing pipeline (Notebook 03).

In [ ]:
# Check for missing values
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
if missing_df.empty:
    print('No NaN values found in raw data (check for whitespace strings in TotalCharges).')
else:
    display(missing_df)

# Check TotalCharges for whitespace/empty strings
print('\n=== TotalCharges whitespace check ===')
if df['TotalCharges'].dtype == object:
    whitespace_mask = df['TotalCharges'].str.strip() == ''
    print(f'Rows with whitespace TotalCharges: {whitespace_mask.sum()}')
    print('These customers are new (tenure=0) and have no total charges yet.')
    display(df[whitespace_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

# Check for duplicate rows
print('\n=== Duplicate Rows ===')
n_duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {n_duplicates}')

# Data types
print('\n=== Data Types ===')
print(df.dtypes)

## Target Variable Analysis

The target variable is `Churn`, which indicates whether a customer left the company within the last month.

- `Yes` = Customer churned
- `No` = Customer stayed

Understanding the class distribution is critical before model training. If the classes are heavily imbalanced, a naive model that always predicts "No" would appear very accurate — but would be useless for identifying churners.

In [ ]:
# Churn value counts and rate
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('=== Churn Distribution ===')
churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage (%)': churn_pct.round(2)
})
display(churn_summary)

churn_rate = churn_pct.get('Yes', 0)
print(f'\nOverall Churn Rate: {churn_rate:.2f}%')
print(f'Churned customers: {churn_counts.get("Yes", 0):,}')
print(f'Retained customers: {churn_counts.get("No", 0):,}')

## Class Imbalance

The dataset is **moderately imbalanced** — roughly 73.5% of customers did NOT churn, while only 26.5% did.

This imbalance means:
- A naive model predicting "No churn" for everyone would achieve ~73.5% accuracy
- We need metrics like **Recall**, **Precision**, **F1-score**, and **ROC-AUC** instead of raw accuracy
- We will use **SMOTE** (Synthetic Minority Over-sampling Technique) during model training to handle this imbalance

The bar chart below visualizes the class distribution.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of churn counts
colors = ['#2ecc71', '#e74c3c']
churn_labels = churn_counts.index.tolist()
churn_values = churn_counts.values.tolist()

bars = axes[0].bar(churn_labels, churn_values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Status', fontsize=12)
axes[0].set_ylabel('Number of Customers', fontsize=12)
axes[0].set_ylim(0, max(churn_values) * 1.15)

# Add count labels on bars
for bar, val in zip(bars, churn_values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{val:,}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

# Pie chart of churn percentages
pct_values = churn_pct.values.tolist()
wedge_colors = ['#2ecc71', '#e74c3c']
explode = [0, 0.05]
axes[1].pie(
    pct_values,
    labels=churn_labels,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=140,
    explode=explode,
    textprops={'fontsize': 12}
)
axes[1].set_title('Churn Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.suptitle('Class Imbalance in Target Variable (Churn)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to ../reports/figures/churn_distribution.png')

## Key Findings

Here is a summary of the data profiling results:

| Metric | Value |
|--------|-------|
| Total customers | 7,043 |
| Total features | 21 columns |
| Churn rate | 26.54% |
| Churned customers | 1,869 |
| Retained customers | 5,174 |
| Missing `TotalCharges` values | 11 (whitespace strings, tenure = 0) |
| Duplicate rows | 0 |
| `TotalCharges` dtype in raw data | object (needs conversion to float) |

### Action Items for Preprocessing
1. Convert `TotalCharges` from string to float (replace whitespace with NaN, then impute)
2. Drop `customerID` (identifier, not a feature)
3. Encode `Churn`: Yes → 1, No → 0
4. Apply SMOTE during model training to handle class imbalance

Proceed to **Notebook 02: EDA** for deeper analysis of individual features.